In [1]:
import pandas as pd

df = pd.read_csv("facebook_ads.csv")

df.head()

,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0
3,708815,916,103928,30-34,M,28,4259,1,1.25,1,0
4,708818,916,103928,30-34,M,28,4133,1,1.29,1,1


In [2]:
# Calculate main marketing KPIs
df['CTR'] = df['Clicks'] / df['Impressions']                # Click-Through Rate
df['CPC'] = df['Spent'] / df['Clicks'].replace(0, 1)         # Cost Per Click
df['CPM'] = (df['Spent'] / df['Impressions']) * 1000         # Cost Per 1000 Impressions
df['CPA'] = df['Spent'] / df['Approved_Conversion'].replace(0, 1)  # Cost Per Acquisition (Approved Conversions)

# Show selected columns including new KPIs
df[['Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion', 
    'CTR', 'CPC', 'CPM', 'CPA']].head()

,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion,CTR,CPC,CPM,CPA
0,7350,1,1.43,2,1,0.000136,1.43,0.194558,1.43
1,17861,2,1.82,2,0,0.000112,0.91,0.101898,1.82
2,693,0,0.00,1,0,0.000000,0.00,0.000000,0.00
3,4259,1,1.25,1,0,0.000235,1.25,0.293496,1.25
4,4133,1,1.29,1,1,0.000242,1.29,0.312122,1.29


In [3]:
# Check for missing values
print("Missing values per column:\n", df.isnull().sum())

# Check for duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Check column data types
print("\nData types:\n", df.dtypes)

Missing values per column:
 ad_id                  0
xyz_campaign_id        0
fb_campaign_id         0
age                    0
gender                 0
interest               0
Impressions            0
Clicks                 0
Spent                  0
Total_Conversion       0
Approved_Conversion    0
CTR                    0
CPC                    0
CPM                    0
CPA                    0
dtype: int64

Number of duplicate rows: 0

Data types:
 ad_id                    int64
xyz_campaign_id          int64
fb_campaign_id           int64
age                     object
gender                  object
interest                 int64
Impressions              int64
Clicks                   int64
Spent                  float64
Total_Conversion         int64
Approved_Conversion      int64
CTR                    float64
CPC                    float64
CPM                    float64
CPA                    float64
dtype: object


In [4]:
# Group by campaign ID and calculate performance metrics
campaign_perf = df.groupby('fb_campaign_id').agg({
    'Impressions': 'sum',
    'Clicks': 'sum',
    'Spent': 'sum',
    'Approved_Conversion': 'sum'
}).reset_index()

# Recalculate KPIs at campaign level
campaign_perf['CTR'] = campaign_perf['Clicks'] / campaign_perf['Impressions']
campaign_perf['CPC'] = campaign_perf['Spent'] / campaign_perf['Clicks'].replace(0, 1)
campaign_perf['CPA'] = campaign_perf['Spent'] / campaign_perf['Approved_Conversion'].replace(0, 1)
campaign_perf['CPM'] = (campaign_perf['Spent'] / campaign_perf['Impressions']) * 1000

campaign_perf.head()

,fb_campaign_id,Impressions,Clicks,Spent,Approved_Conversion,CTR,CPC,CPA,CPM
0,103916,7350,1,1.43,1,0.000136,1.43,1.43,0.194558
1,103917,17861,2,1.82,0,0.000112,0.91,1.82,0.101898
2,103920,693,0,0.00,0,0.000000,0.00,0.00,0.000000
3,103928,8392,2,2.54,1,0.000238,1.27,2.54,0.302669
4,103929,1915,0,0.00,1,0.000000,0.00,0.00,0.000000


In [5]:
# Best campaign by CPA (lowest cost per acquisition)
best_cpa = campaign_perf[campaign_perf['Approved_Conversion'] > 0].sort_values('CPA').head(1)

# Worst campaign by CPA (highest cost)
worst_cpa = campaign_perf[campaign_perf['Approved_Conversion'] > 0].sort_values('CPA', ascending=False).head(1)

# Best campaign by CTR
best_ctr = campaign_perf.sort_values('CTR', ascending=False).head(1)

# Campaigns with spend but zero approved conversions
wasted_spend = campaign_perf[(campaign_perf['Spent'] > 0) & (campaign_perf['Approved_Conversion'] == 0)]

best_cpa, worst_cpa, best_ctr, wasted_spend.head()

(     fb_campaign_id  Impressions  Clicks  Spent  Approved_Conversion  CTR  \
 383          123639          784       0    0.0                    1  0.0   
 
      CPC  CPA  CPM  
 383  0.0  0.0  0.0  ,
      fb_campaign_id  Impressions  Clicks   Spent  Approved_Conversion  \
 594          144742      1554494     401  569.53                    1   
 
           CTR       CPC     CPA       CPM  
 594  0.000258  1.420274  569.53  0.366376  ,
      fb_campaign_id  Impressions  Clicks  Spent  Approved_Conversion  \
 132          109857          944       1   1.42                    0   
 
           CTR   CPC   CPA       CPM  
 132  0.001059  1.42  1.42  1.504237  ,
     fb_campaign_id  Impressions  Clicks  Spent  Approved_Conversion       CTR  \
 1           103917        17861       2   1.82                    0  0.000112   
 5           103940        15615       3   4.77                    0  0.000192   
 7           103951         2355       1   1.50                    0  0.000425   
 

In [6]:
# Group performance by Age
age_perf = df.groupby('age').agg({
    'Impressions': 'sum',
    'Clicks': 'sum',
    'Spent': 'sum',
    'Approved_Conversion': 'sum'
}).reset_index()

# Recalculate KPIs for each age group
age_perf['CTR'] = age_perf['Clicks'] / age_perf['Impressions']
age_perf['CPC'] = age_perf['Spent'] / age_perf['Clicks'].replace(0, 1)
age_perf['CPA'] = age_perf['Spent'] / age_perf['Approved_Conversion'].replace(0, 1)
age_perf['CPM'] = (age_perf['Spent'] / age_perf['Impressions']) * 1000

age_perf

,age,Impressions,Clicks,Spent,Approved_Conversion,CTR,CPC,CPA,CPM
0,30-34,67993019,9483,15252.399986,494,0.000139,1.608394,30.875304,0.224323
1,35-39,42104644,7094,11112.429994,207,0.000168,1.566455,53.683237,0.263924
2,40-44,39604307,7736,11589.729981,170,0.000195,1.498155,68.174882,0.292638
3,45-49,63732858,13852,20750.669997,208,0.000217,1.498027,99.762837,0.325588


In [7]:
# Group performance by Gender
gender_perf = df.groupby('gender').agg({
    'Impressions': 'sum',
    'Clicks': 'sum',
    'Spent': 'sum',
    'Approved_Conversion': 'sum'
}).reset_index()

# Recalculate KPIs for gender groups
gender_perf['CTR'] = gender_perf['Clicks'] / gender_perf['Impressions']
gender_perf['CPC'] = gender_perf['Spent'] / gender_perf['Clicks'].replace(0, 1)
gender_perf['CPA'] = gender_perf['Spent'] / gender_perf['Approved_Conversion'].replace(0, 1)
gender_perf['CPM'] = (gender_perf['Spent'] / gender_perf['Impressions']) * 1000

gender_perf

,gender,Impressions,Clicks,Spent,Approved_Conversion,CTR,CPC,CPA,CPM
0,F,114862847,23878,34502.619963,495,0.000208,1.444954,69.702263,0.300381
1,M,98571981,14287,24202.609995,584,0.000145,1.694030,41.442825,0.245532


In [8]:
# Prepare the dataset for Power BI by selecting the most important columns

powerbi_df = df[[
    'ad_id',
    'xyz_campaign_id',
    'fb_campaign_id',
    'age',
    'gender',
    'interest',
    'Impressions',
    'Clicks',
    'Spent',
    'Total_Conversion',
    'Approved_Conversion',
    'CTR',
    'CPC',
    'CPM',
    'CPA'
]]

# Save as CSV file
powerbi_df.to_csv("marketing_powerbi_ready.csv", index=False)

powerbi_df.head()


,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion,CTR,CPC,CPM,CPA
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1,0.000136,1.43,0.194558,1.43
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0,0.000112,0.91,0.101898,1.82
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0,0.000000,0.00,0.000000,0.00
3,708815,916,103928,30-34,M,28,4259,1,1.25,1,0,0.000235,1.25,0.293496,1.25
4,708818,916,103928,30-34,M,28,4133,1,1.29,1,1,0.000242,1.29,0.312122,1.29
